<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/Task_4B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class PyTorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
        super(PyTorchMLP, self).__init__()

        #Define sequential layer architecture
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.Tanh(),
            nn.Linear(hidden_dim2, output_dim)
            #Softmax is omitted here because nn.CrossEntropyLoss handles it internally
        )

    def forward(self, x):
        """Executes the automatic computational graph forward pass."""
        return self.network(x)
if __name__ == "__main__":
    # Fix random seed for reproducibility
    torch.manual_seed(42)

    # 1. Generate Synthetic Data (100 samples, 4 features)
    X_raw = torch.randn(100, 4)
    # Target indices for CrossEntropyLoss (long integer format, not one-hot encoded)
    Y_raw = torch.randint(0, 3, (100,))

    #Package into a Mini-Batch Loader
    dataset = TensorDataset(X_raw, Y_raw)
    data_loader = DataLoader(dataset, batch_size=16, shuffle=True)

    #Instantiate Network, Loss Function, and Optimizer
    model = PyTorchMLP(input_dim=4, hidden_dim1=8, hidden_dim2=6, output_dim=3)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1)

    #Execute Mini-Batch Training Loop
    epochs = 100
    print("Starting PyTorch Mini-Batch Training:")

    for epoch in range(epochs):
        epoch_loss = 0.0

        for X_batch, Y_batch in data_loader:
            #Clear historical tracking gradients
            optimizer.zero_grad()

            #Forward Pass
            predictions = model(X_batch)

            #Compute Loss
            loss = criterion(predictions, Y_batch)

            #Backpropagation (PyTorch Autograd engine calculates analytical solutions)
            loss.backward()

            #Update Parameters via SGD optimization mechanics
            optimizer.step()

            epoch_loss += loss.item() * X_batch.size(0)

        epoch_loss /= len(data_loader.dataset)

        #Status update logging
        if (epoch + 1) % (epochs // 10) == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d}/{epochs:03d} -> Loss: {epoch_loss:.4f}")


Starting PyTorch Mini-Batch Training:
Epoch 001/100 -> Loss: 1.1600
Epoch 010/100 -> Loss: 1.0605
Epoch 020/100 -> Loss: 1.0144
Epoch 030/100 -> Loss: 1.0079
Epoch 040/100 -> Loss: 0.9904
Epoch 050/100 -> Loss: 0.9831
Epoch 060/100 -> Loss: 0.9817
Epoch 070/100 -> Loss: 0.9709
Epoch 080/100 -> Loss: 0.9545
Epoch 090/100 -> Loss: 0.9467
Epoch 100/100 -> Loss: 0.9317


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class PyTorchCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(PyTorchCNN, self).__init__()

        #Feature Extraction Layers (Convolutional & Pooling)
        self.features = nn.Sequential(
            #Input: (Batch, 3, 32, 32)
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            #Output: (Batch, 16, 32, 32)

            nn.MaxPool2d(kernel_size=2, stride=2),
            #Output: (Batch, 16, 16, 16)

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            #Output: (Batch, 32, 16, 16)

            nn.MaxPool2d(kernel_size=2, stride=2)
            #Output: (Batch, 32, 8, 8)
        )

        #Fully Connected Classifier Layers
        #Spatial size is reduced from 32x32 to 8x8, with 32 output channels
        self.classifier = nn.Sequential(
            nn.Flatten(), #Flattens (Batch, 32, 8, 8) -> (Batch, 32 * 8 * 8)
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes) #Raw logits (Softmax is handled by nn.CrossEntropyLoss)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
if __name__ == "__main__":
    torch.manual_seed(42)

    #Generate Synthetic Image Data (64 images, 3 channels, 32x32 pixels)
    X_images = torch.randn(64, 3, 32, 32)
    Y_labels = torch.randint(0, 10, (64,)) #10 target classes

    #Package into a Loader
    dataset = TensorDataset(X_images, Y_labels)
    data_loader = DataLoader(dataset, batch_size=16, shuffle=True)

    #Instantiate Architecture
    model = PyTorchCNN(num_classes=10)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001) #Using Adam optimizer

    #Training Loop
    epochs = 5
    print("Starting PyTorch CNN Training Loop:")

    for epoch in range(epochs):
        epoch_loss = 0.0

        for X_batch, Y_batch in data_loader:
            optimizer.zero_grad()

            #Forward pass
            predictions = model(X_batch)

            #Loss calculation
            loss = criterion(predictions, Y_batch)

            #Backward pass & weight updates
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * X_batch.size(0)

        epoch_loss /= len(data_loader.dataset)
        print(f"Epoch {epoch+1}/{epochs} -> Loss: {epoch_loss:.4f}")

    print("\nStatus: CNN forward and backward pipelines executed successfully.")


Starting PyTorch CNN Training Loop:
Epoch 1/5 -> Loss: 2.4082
Epoch 2/5 -> Loss: 2.2669
Epoch 3/5 -> Loss: 2.2281
Epoch 4/5 -> Loss: 2.1469
Epoch 5/5 -> Loss: 2.0433

Status: CNN forward and backward pipelines executed successfully.


In [10]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Initialize dummy in-memory tensors
X_data = torch.randn(1000, 3, 32, 32)  # 1000 images, 3 channels, 32x32 pixels
Y_data = torch.randint(0, 10, (1000,)) # 10 target classes

# 2. Bind them into a structured Dataset object
memory_dataset = TensorDataset(X_data, Y_data)

# 3. Instantiate the high-performance DataLoader
train_loader = DataLoader(
    dataset=memory_dataset,
    batch_size=32,            # Group features into subsets of 32
    shuffle=True,             # Randomise data indexing every epoch
    num_workers=2,            # Subcontract batch assembly to 2 parallel CPU worker subprocesses
    drop_last=True            # Discard any uneven final batch (if 1000 % 32 != 0)
)
import numpy as np
from torch.utils.data import Dataset

class CustomImageDataset(Dataset):
    """A map-style dataset mimicking real-world production image ingestion."""
    def __init__(self, num_samples=500):
        # In a real app, you would save file paths here: self.file_paths = [...]
        self.num_samples = num_samples

    def __len__(self):
        """Mandatory: Returns the exact global pool limit size."""
        return self.num_samples

    def __getitem__(self, idx):
        """Mandatory: Reads and returns exactly one specific data sample pair."""
        # Simulated disk read (e.g., loading an image from disk using OpenCV/PIL)
        mock_image_file = np.random.rand(3, 32, 32).astype(np.float32)
        mock_label_file = int(np.random.randint(0, 10))

        # Convert lazy-loaded data matrices into PyTorch tensors
        x_tensor = torch.from_numpy(mock_image_file)
        y_tensor = torch.tensor(mock_label_file, dtype=torch.long)

        return x_tensor, y_tensor
if __name__ == "__main__":
    # 1. Instantiate custom structural map dataset
    disk_dataset = CustomImageDataset(num_samples=100)

    # 2. Initialize DataLoader
    custom_loader = DataLoader(
        dataset=disk_dataset,
        batch_size=16,
        shuffle=True,
        num_workers=0,        # Set to 0 if executing code inside interactive environments like Jupyter notebooks
        pin_memory=True       # Speeds up tensor host-to-device transfers when training on GPUs
    )

    # 3. Method A: Inspect a single sample batch manually
    data_iter = iter(custom_loader)
    first_images, first_labels = next(data_iter)

    print("--- Single Mini-Batch Dimensional Review ---")
    print("Batch Images Shape :", first_images.shape)  # Expected: [16, 3, 32, 32]
    print("Batch Labels Shape :", first_labels.shape)  # Expected: [16]

    # 4. Method B: Standard operational training epoch loop integration
    print("\n--- Iterating Through Full Data Loader ---")
    for batch_idx, (X_batch, Y_batch) in enumerate(custom_loader):
        # This is where you pass X_batch to your model: predictions = model(X_batch)
        if (batch_idx + 1) % 3 == 0:
            print(f"Processed Mini-Batch {batch_idx + 1:02d} | Ingested Samples: {len(X_batch) * (batch_idx + 1)}")


--- Single Mini-Batch Dimensional Review ---
Batch Images Shape : torch.Size([16, 3, 32, 32])
Batch Labels Shape : torch.Size([16])

--- Iterating Through Full Data Loader ---
Processed Mini-Batch 03 | Ingested Samples: 48
Processed Mini-Batch 06 | Ingested Samples: 96


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class RegularizedCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(RegularizedCNN, self).__init__()

        # 1. Feature Extraction Layers
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 2. Classifier Layers with Dropout Regularization
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),

            # During training, randomly zero out 50% of the activations
            nn.Dropout(p=0.5),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
if __name__ == "__main__":
    torch.manual_seed(42)

    # 1. Generate Synthetic Data
    X_data = torch.randn(64, 3, 32, 32)
    Y_data = torch.randint(0, 10, (64,))

    loader = DataLoader(TensorDataset(X_data, Y_data), batch_size=16, shuffle=True)

    # 2. Instantiate Network
    model = RegularizedCNN(num_classes=10)
    criterion = nn.CrossEntropyLoss()

    # 3. Apply Weight Decay (L2 Penalization) in the Optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4  # L2 regularization strength parameter
    )

    # --- TRAINING MODE ---
    model.train()  # Activates Dropout layers globally
    print("--- Training Execution (Dropout Active) ---")

    for epoch in range(3):
        for X_batch, Y_batch in loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), Y_batch)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} Complete | Loss: {loss.item():.4f}")

    # --- INFERENCE MODE ---
    model.eval()  # Disables Dropout layers globally
    print("\n--- Inference Execution (Dropout Disabled) ---")

    with torch.no_grad():  # Disables gradient tracking graph updates
        test_sample = torch.randn(1, 3, 32, 32)
        raw_output = model(test_sample)
        print("Model Logits Shape:", raw_output.shape)


--- Training Execution (Dropout Active) ---
Epoch 1 Complete | Loss: 2.3434
Epoch 2 Complete | Loss: 2.2690
Epoch 3 Complete | Loss: 2.2884

--- Inference Execution (Dropout Disabled) ---
Model Logits Shape: torch.Size([1, 10])


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

#Simple architecture for demonstration
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 3)
        )
    def forward(self, x):
        return self.net(x)

if __name__ == "__main__":
    torch.manual_seed(42)

    #Generate mock dataset (128 samples, 4 features, 3 target classes)
    X_data = torch.randn(128, 4)
    Y_data = torch.randint(0, 3, (128,))
    loader = DataLoader(TensorDataset(X_data, Y_data), batch_size=32, shuffle=True)

    #Instantiate Core Network, Loss, and Base Optimizer
    model = SimpleMLP()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1) #Starting Learning Rate = 0.1[the standard starting learning rate]

    #Configure Schedulers (Choose one pattern for your specific use-case)

    #Step Decay (Reduces LR by multiplying by 'gamma' every 'step_size' epochs)
    step_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

    #Adaptive Plateau Decay (Reduces LR when a monitored metric stops improving)
    plateau_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

    #Execution Training Loop
    epochs = 6
    print("--- Starting Training Loop with LR Schedulers ---")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, Y_batch in loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), Y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * X_batch.size(0)

        epoch_loss /= len(loader.dataset)

        #Fetch the current active learning rate for logging
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Avg Loss: {epoch_loss:.4f} | Active LR: {current_lr:.5f}")

        #Advance the Schedulers
        #StepLR updates strictly based on the epoch count
        step_scheduler.step()

        #ReduceLROnPlateau requires the loss metric to judge if a plateau has occurred
        plateau_scheduler.step(epoch_loss)


--- Starting Training Loop with LR Schedulers ---
Epoch 01/06 | Avg Loss: 1.1391 | Active LR: 0.10000
Epoch 02/06 | Avg Loss: 1.1264 | Active LR: 0.10000
Epoch 03/06 | Avg Loss: 1.1190 | Active LR: 0.05000
Epoch 04/06 | Avg Loss: 1.1160 | Active LR: 0.05000
Epoch 05/06 | Avg Loss: 1.1126 | Active LR: 0.02500
Epoch 06/06 | Avg Loss: 1.1115 | Active LR: 0.02500
